# ResearchGPT — Colab Experimentation Notebook
Quick, self-contained RAG pipeline you can run end-to-end in Colab before wiring it into the full FastAPI + Streamlit app.

**Steps:** install deps → set Groq API key → upload a PDF → build index → ask questions with citations.

In [6]:
!pip install -q langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu pymupdf groq langchain-text-splitters

In [2]:
import os
from getpass import getpass

# Get a free key at https://console.groq.com/keys
os.environ['GROQ_API_KEY'] = getpass('Enter your Groq API key: ')

Enter your Groq API key: ··········


In [3]:
from google.colab import files
uploaded = files.upload()  # select one or more PDF papers
pdf_paths = list(uploaded.keys())
print('Uploaded:', pdf_paths)

Saving research paper.pdf to research paper.pdf
Uploaded: ['research paper.pdf']


In [4]:
import fitz  # PyMuPDF
from dataclasses import dataclass

@dataclass
class PageContent:
    doc_name: str
    page_number: int
    text: str

def extract_pages(pdf_path):
    doc_name = pdf_path.split('/')[-1]
    pages = []
    with fitz.open(pdf_path) as doc:
        for i, page in enumerate(doc):
            text = page.get_text('text').strip()
            if text:
                pages.append(PageContent(doc_name, i + 1, text))
    return pages

all_pages = []
for p in pdf_paths:
    all_pages.extend(extract_pages(p))

print(f'Extracted {len(all_pages)} non-empty pages')

Extracted 11 non-empty pages


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import uuid

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150,
                                           separators=['\n\n', '\n', '. ', ' ', ''])

documents = []
for page in all_pages:
    pieces = splitter.split_text(page.text)
    for piece in pieces:
        documents.append(Document(page_content=piece,
                                    metadata={'doc_name': page.doc_name, 'page_number': page.page_number}))

print(f'Created {len(documents)} chunks')

Created 97 chunks


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2',
                                     encode_kwargs={'normalize_embeddings': True})

vector_store = FAISS.from_documents(documents, embeddings)
print('Vector index built with', vector_store.index.ntotal, 'vectors')

/tmp/ipykernel_5206/1158409622.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector index built with 97 vectors


In [10]:
from groq import Groq

client = Groq(api_key=os.environ['GROQ_API_KEY'])

PROMPT_TEMPLATE = '''You are ResearchGPT, an assistant that answers questions strictly using the provided research paper excerpts.
Only use information found in the CONTEXT. If the answer isn't there, say so honestly. Be concise.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:'''

def ask(question, k=4):
    results = vector_store.similarity_search_with_relevance_scores(question, k=k)
    context = '\n\n'.join(
        f"[Source {i+1} | {doc.metadata['doc_name']} | Page {doc.metadata['page_number']}]\n{doc.page_content}"
        for i, (doc, score) in enumerate(results)
    )
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2,
        max_tokens=800,
    )
    answer = response.choices[0].message.content.strip()

    print('ANSWER:\n', answer)
    print('\nCITATIONS:')
    for doc, score in results:
        print(f"  - {doc.metadata['doc_name']} (page {doc.metadata['page_number']}, relevance {score:.3f})")
    return answer, results

In [11]:
# Try it out
_ = ask('What problem does this paper try to solve, and what method do they propose?')

/tmp/ipykernel_5206/3210877950.py:16: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='2f5d4930-2fbb-4236-8fd1-8a322d93ab01', metadata={'doc_name': 'research paper.pdf', 'page_number': 4}, page_content='is to minimize the log-probability that the discriminator correctly\ndistinguishes samples from G. Because the sampling of s is discrete,\nwe follow [21, 25, 33] to compute the gradient of V (G, D) with\nrespect to θG by policy gradient:\n∇θG V (G, D) = ∇θG\nV\nÕ\nc=1\nEs∼G(·|vc )[log(1 −D(s))]\n=\nV\nÕ\nc=1\nEs∼G(·|vc )[∇θG logG(s |vc) log(1 −D(s))].\n(3)\n4.3\nA Naive Implementation of D and G\nA naive implementation of the discriminator and generator is based\non sigmoid and softmax functions, respectively.\nFor the discriminator D, intuitively we can define it as the multi-\nplication of the sigmoid function of the inner product of every two\nvertices in the input vertex subset s:\nD(s) =\nÖ\n(u,v)∈s,u,v\nσ(d⊤\nu · dv ),\n(4)\nwhere du, dv ∈Rk are the k-dime

ANSWER:
 Based on the provided excerpts, the paper tries to solve the problem of community detection and motif generation in complex networks. 

The proposed method is called CommunityGAN, which is a generative adversarial network (GAN) framework that combines community detection and motif generation.

CITATIONS:
  - research paper.pdf (page 4, relevance -0.005)
  - research paper.pdf (page 4, relevance -0.023)
  - research paper.pdf (page 6, relevance -0.041)
  - research paper.pdf (page 10, relevance -0.042)


In [12]:
_ = ask("What dataset did they use?")
_ = ask("What were the main results?")
_ = ask("What are the limitations mentioned by the authors?")

/tmp/ipykernel_5206/3210877950.py:16: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='fef7738a-0079-4abb-81e3-761fdf16484e', metadata={'doc_name': 'research paper.pdf', 'page_number': 9}, page_content='0\n20\n40\n60\n80\n100\nIteration\n0.0\n0.1\n0.2\nThe increase in F1-Score\nAmazon\nYoutube\nDBLP\n(a) Generator\n0\n20\n40\n60\n80\n100\nIteration\n-0.1\n0.0\n0.1\nThe increase in F1-Score\nAmazon\nYoutube\nDBLP\n(b) Discriminator\nFigure 8: Learning curves of the generator and the discrimi-\nnator of CommunityGAN in community detection.\nto quantify the accuracy of community detection methods by eval-\nuating the level of correspondence between detected and ground-\ntruth communities.\nSetup. In the community detection experiment, all the baselines\nand setups are the same as the synthetic data experiments, whose\ndetails refer to §5.4.\nEvaluation Metric. Besides the F1-Score described in synthetic\ndataset experiments (refer to §5.1), to comprehensively eval

ANSWER:
 The datasets used are:

1. Amazon
2. Youtube
3. DBLP
4. arXiv-AstroPh
5. arXiv-GrQc

CITATIONS:
  - research paper.pdf (page 9, relevance 0.087)
  - research paper.pdf (page 9, relevance 0.018)
  - research paper.pdf (page 9, relevance -0.010)
  - research paper.pdf (page 2, relevance -0.035)


/tmp/ipykernel_5206/3210877950.py:16: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='5bc4d8e8-7c77-4153-a96c-5f92016f8928', metadata={'doc_name': 'research paper.pdf', 'page_number': 9}, page_content='F1-Score on the three datasets differs very much, we only demon-\nstrate the relative increase in the score. As we can see, the generator\nperforms outstandingly well after convergence, while the perfor-\nmance of the discriminator falls a little or boosts at first and then\nfalls. Maybe it is because that the discriminator aims to distinguish'), np.float32(-0.015032887)), (Document(id='fef7738a-0079-4abb-81e3-761fdf16484e', metadata={'doc_name': 'research paper.pdf', 'page_number': 9}, page_content='0\n20\n40\n60\n80\n100\nIteration\n0.0\n0.1\n0.2\nThe increase in F1-Score\nAmazon\nYoutube\nDBLP\n(a) Generator\n0\n20\n40\n60\n80\n100\nIteration\n-0.1\n0.0\n0.1\nThe increase in F1-Score\nAmazon\nYoutube\nDBLP\n(b) Discriminator\nFigure 8: Learning curves of the 

ANSWER:
 Based on the provided research paper excerpts, the main results are as follows:

- The CommunityGAN model achieved the highest F1-Score on the community detection experiment, outperforming other models on all three datasets (Amazon, Youtube, and DBLP).
- The F1-Score of the CommunityGAN model on the Amazon dataset was 0.860, on the Youtube dataset was 0.327, and on the DBLP dataset was 0.456.
- The NMI (Normalized Mutual Information) of the CommunityGAN model on the Amazon dataset was 0.853, on the Youtube dataset was 0.091, and on the DBLP dataset was 0.153.
- The learning curves of the generator and discriminator in the CommunityGAN model are shown in Figure 8, indicating that the generator performs well after convergence, while the discriminator's performance falls or boosts initially and then falls.

CITATIONS:
  - research paper.pdf (page 9, relevance -0.015)
  - research paper.pdf (page 9, relevance -0.063)
  - research paper.pdf (page 9, relevance -0.107)
  - research p

/tmp/ipykernel_5206/3210877950.py:16: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='b79c86fd-d40a-4143-88b5-e4287b9a2644', metadata={'doc_name': 'research paper.pdf', 'page_number': 9}, page_content='are defined by the product categories in Amazon. This graph\nhas 3,225 vertices and 10,262 edges.\nYoutube is a network of social relationships of Youtube web-\nsite users. The vertices represent users; the edges indicate\nfriendships among the users; the user-defined groups are\nconsidered as ground-truth communities. This graph has\n4,890 vertices and 20,787 edges.\nDBLP is a co-authorship network from DBLP. The vertices\nrepresent researchers; the edges indicate the co-author rela-\ntionships; authors who have published in a same journal or\nconference form a community. This graph has 10,824 vertices\nand 38,732 edges.\nDatasets without ground-truth communities.3\narXiv-AstroPh is from the e-print arXiv and covers scientific\ncollaborations between authors wi

ANSWER:
 Unfortunately, the provided research paper excerpts do not mention the limitations of the CommunityGAN method.

CITATIONS:
  - research paper.pdf (page 9, relevance -0.107)
  - research paper.pdf (page 9, relevance -0.111)
  - research paper.pdf (page 11, relevance -0.155)
  - research paper.pdf (page 10, relevance -0.159)
